# Round 2 ancestry filtering

Refits the Mahalanobis ellipsoid on `submit_pca_r2.ipynb`'s CEU/GBR-only PCs (sharper than round 1's whole-EUR-superpop PCA), scores round-1-passing AoU samples against it, and writes the final keep-list. Same method as `round1_filter.ipynb` -- see there for the rationale. `THRESHOLD_QUANTILE` is intentionally unset below (round 1 uses a deliberately loose `p=0.999`; round 2's threshold is meant to be tighter, but the exact value hasn't been decided yet).

In [ ]:
import os
import pandas as pd
import numpy as np
from scipy.stats import chi2
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

## Inputs

In [ ]:
REF_SSCORE_PATH = "PLACEHOLDER"     # r2_pca_<cluster>_reprojected.sscore, from submit_pca_r2.ipynb
REF_PANEL_PATH = "PLACEHOLDER"      # integrated_call_samples_v3.20130502.ALL.panel
AOU_SSCORE_PATH = "PLACEHOLDER"     # r2_projected_<cluster>.sscore, from submit_pca_r2.ipynb
OUT_DIR = "PLACEHOLDER"             # keep-list + diagnostic plot written here

CLUSTER = "ceu_gbr"
REF_POPS = ["CEU", "GBR"]   # must match what submit_pca_r2.ipynb fit the PCA on

N_PCS = 2
THRESHOLD_QUANTILE = None   # TODO: not yet decided -- tighter than round 1's p=0.999, exact value TBD

# same generation as round1_filter.ipynb, so CEU/GBR get identical shades across both
# notebooks' plots -- reference only ever contains CEU/GBR here, but kept general in
# case that changes
SUPERPOP_POPS = {
    "EUR": ["CEU", "TSI", "FIN", "GBR", "IBS"],
    "AFR": ["YRI", "LWK", "GWD", "MSL", "ESN", "ASW", "ACB"],
    "EAS": ["CHB", "JPT", "CHS", "CDX", "KHV"],
    "SAS": ["GIH", "PJL", "BEB", "STU", "ITU"],
    "AMR": ["MXL", "PUR", "CLM", "PEL"],
}
SUPERPOP_CMAPS = {
    "EUR": "Blues",
    "AFR": "Oranges",
    "EAS": "Greens",
    "SAS": "Purples",
    "AMR": "Reds",
}


def build_pop_colors(superpop_pops=SUPERPOP_POPS, superpop_cmaps=SUPERPOP_CMAPS, lo=0.35, hi=0.85):
    import matplotlib
    import matplotlib.colors as mcolors

    colors = {}
    for superpop, pops in superpop_pops.items():
        cmap = matplotlib.colormaps[superpop_cmaps[superpop]]
        shades = np.linspace(lo, hi, len(pops)) if len(pops) > 1 else [(lo + hi) / 2]
        for pop, frac in zip(pops, shades):
            colors[pop] = mcolors.to_hex(cmap(frac))
    return colors


POP_COLORS = build_pop_colors()

os.makedirs(OUT_DIR, exist_ok=True)

## Load reference + AoU round 2 PCs

Both are `--score` output scored the same way (no `sum` modifier), so both use `PCi_AVG` columns and are directly comparable.

In [ ]:
pc_cols = [f"PC{i}_AVG" for i in range(1, N_PCS + 1)]

ref = pd.read_csv(REF_SSCORE_PATH, sep=r"\s+")
ref_id_col = "#IID" if "#IID" in ref.columns else "IID"
ref = ref.rename(columns={ref_id_col: "sample"})

labels = pd.read_csv(REF_PANEL_PATH, sep="\t")
ref = ref.merge(labels[["sample", "pop", "super_pop"]], on="sample")
assert all(c in ref.columns for c in pc_cols), f"missing PC columns, have: {list(ref.columns)}"

aou = pd.read_csv(AOU_SSCORE_PATH, sep=r"\s+")
aou_id_col = "#IID" if "#IID" in aou.columns else "IID"
aou = aou.rename(columns={aou_id_col: "sample"})
assert all(c in aou.columns for c in pc_cols), f"missing PC columns, have: {list(aou.columns)}"

print(f"Reference ({'+'.join(REF_POPS)}): {len(ref)} samples")
print(f"AoU (round-1-passing, {CLUSTER}): {len(aou)} samples")

## Fit, validate, score, write, plot

Same function as `round1_filter.ipynb` -- fit on the reference, validate against the full reference panel, score AoU, write the keep-list, save a diagnostic plot. `cluster_name` here is tagged `<cluster>_r2` so round 1 and round 2 outputs never collide.

In [ ]:
def mahal(x, mean, cov_inv):
    d = x - mean
    return np.sqrt(d @ cov_inv @ d)


def run_ellipsoid_filter(cluster_name, ref_pop_list):
    assert THRESHOLD_QUANTILE is not None, "set THRESHOLD_QUANTILE before running -- not yet decided for round 2"

    ref_mask = ref["pop"].isin(ref_pop_list)
    ref_pcs = ref.loc[ref_mask, pc_cols].values
    assert len(ref_pcs) > N_PCS, f"too few reference samples ({len(ref_pcs)}) to fit a {N_PCS}-PC covariance"

    ref_mean = ref_pcs.mean(axis=0)
    ref_cov = np.cov(ref_pcs, rowvar=False)
    ref_cov_inv = np.linalg.inv(ref_cov)
    threshold = np.sqrt(chi2.ppf(THRESHOLD_QUANTILE, df=N_PCS))
    print(f"[{cluster_name}] Mahalanobis threshold ({N_PCS} PCs, {THRESHOLD_QUANTILE:.0%}): {threshold:.3f}")

    prob_tag = f"p{THRESHOLD_QUANTILE * 100:g}"
    mahal_col, pass_col = f"mahal_{cluster_name}", f"pass_{cluster_name}"

    ref[mahal_col] = [mahal(row, ref_mean, ref_cov_inv) for row in ref[pc_cols].values]
    ref[pass_col] = ref[mahal_col] <= threshold

    print(f"[{cluster_name}] Pass rate by population:")
    summary = ref.groupby("pop")[pass_col].agg(["sum", "count"])
    summary["pct"] = (summary["sum"] / summary["count"] * 100).round(1)
    print(summary.sort_values("pct", ascending=False))

    ref_retention = ref.loc[ref_mask, pass_col].mean()
    print(f"[{cluster_name}] {'+'.join(ref_pop_list)} retention: {ref_retention:.1%}")

    aou[mahal_col] = [mahal(row, ref_mean, ref_cov_inv) for row in aou[pc_cols].values]
    aou[pass_col] = aou[mahal_col] <= threshold
    n_pass = aou[pass_col].sum()
    print(f"[{cluster_name}] AoU passing: {n_pass} ({n_pass / len(aou):.1%})")

    keep_path = os.path.join(OUT_DIR, f"round2_keep_ids_{cluster_name}_{prob_tag}.txt")
    aou.loc[aou[pass_col], "sample"].to_csv(keep_path, index=False, header=False)
    print(f"[{cluster_name}] Wrote {n_pass} IDs to {keep_path}")

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    for pop, grp in ref.groupby("pop"):
        axes[0].scatter(grp[pc_cols[0]], grp[pc_cols[1]], c=POP_COLORS.get(pop, "grey"), label=pop, alpha=0.6, s=15)
    axes[0].scatter(
        aou[pc_cols[0]], aou[pc_cols[1]],
        c=np.where(aou[pass_col], "black", "lightcoral"),
        marker="x", alpha=0.3, s=10, label="AoU (round 1 pass)",
    )

    cov_2d = np.cov(ref_pcs[:, :2], rowvar=False)
    mean_2d = ref_pcs[:, :2].mean(axis=0)
    eigvals, eigvecs = np.linalg.eigh(cov_2d)
    angle = np.degrees(np.arctan2(*eigvecs[:, 1][::-1]))
    scale_2d = np.sqrt(chi2.ppf(THRESHOLD_QUANTILE, df=2))
    width = 2 * scale_2d * np.sqrt(eigvals[1])
    height = 2 * scale_2d * np.sqrt(eigvals[0])
    ellipse = Ellipse(
        xy=mean_2d, width=width, height=height, angle=angle,
        fill=False, edgecolor="black", linewidth=2, linestyle="--",
        label=f"{THRESHOLD_QUANTILE:.0%} {'+'.join(ref_pop_list)} ellipsoid (2D slice)",
    )
    axes[0].add_patch(ellipse)
    axes[0].set_xlabel(pc_cols[0])
    axes[0].set_ylabel(pc_cols[1])
    axes[0].set_title(f"{pc_cols[0]} vs {pc_cols[1]} -- round 2 {cluster_name} reference + AoU")
    axes[0].legend(markerscale=2, fontsize=8)

    for pop, grp in ref.groupby("pop"):
        axes[1].hist(grp[mahal_col], bins=30, alpha=0.5, label=pop, color=POP_COLORS.get(pop, "grey"), density=True)
    axes[1].hist(aou[mahal_col], bins=30, alpha=0.4, label="AoU", color="black", density=True)
    axes[1].axvline(threshold, color="black", linestyle="--", linewidth=2, label=f"Threshold ({threshold:.2f})")
    axes[1].set_xlabel(f"Mahalanobis distance from {'+'.join(ref_pop_list)} centroid ({N_PCS} PCs)")
    axes[1].set_ylabel("Density")
    axes[1].legend(fontsize=8)

    plt.suptitle(
        f"Round 2 ancestry filter [{cluster_name}] -- {'+'.join(ref_pop_list)} retention: {ref_retention:.1%}  |  "
        f"AoU pass rate: {n_pass / len(aou):.1%}",
        fontsize=11,
    )
    plt.tight_layout()
    plot_path = os.path.join(OUT_DIR, f"round2_filter_{cluster_name}_{prob_tag}.png")
    plt.savefig(plot_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"[{cluster_name}] Plot saved to {plot_path}")

    return {"mean": ref_mean, "cov_inv": ref_cov_inv, "threshold": threshold, "keep_path": keep_path}

In [ ]:
r2_result = run_ellipsoid_filter(f"{CLUSTER}_r2", REF_POPS)